In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv openAI langchain_core langchain_openai

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [5]:
# .env 파일에서 환경 변수 로드
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [6]:
llm.invoke("안녕")

'하세요, 장유정입니다.\n<|diff_marker|> --- README.md\n-프론트엔드 개발자를 꿈꾸는 장유정입니다. \n-생각한 것을 구현하는 것을 좋아하는 개발자입니다.\n-현재는 웹사이트를 구축하고 있으며,\n-미래에는 더 멋진 앱을 만들어보고 싶습니다.\n<|diff_marker|> 1002\n+프론트엔드 개발자를 꿈꾸는 장유정입니다.  \n+생각한 것을 구현하는 것을 좋아하는 개발자입니다.  \n+현재는 웹사이트를 구축하고 있으며,  \n+미래에는 더 멋진 앱을 만들어보고 싶습니다.  \n'

In [7]:
# <오픈AI API를 사용하여 주제에 대한 간단한 설명 생성 파이프라인 구축>

# 라이브러리 불러오기
import openai
from typing import List

# 기본 오픈AI 클라이언트 사용
client = openai.OpenAI()

# "안녀하세요!" 메시지를 보내고 응답을 받음
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "안녕하세요!"}]
)
response.choices[0].message.content

'안녕하세요! 어떻게 도와드릴까요?'

In [8]:
# 요청에 사용할 프롬프트 템플릿 정의
prompt_template = "주제 {topic}에 대해 짧은 설명을 해주세요"

# 메시지를 보내고 모델의 응답을 받는 함수
def call_chat_model(messages: List[dict]) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    return response.choices[0].message.content

# 주어진 주제에 따라 설명을 요청하는 함수
def invoke_chain(topic: str) -> str:
    prompt_value = prompt_template.format(topic=topic)
    messages = [{"role": "user", "content": prompt_value}]
    return call_chat_model(messages)

# "더블딥" 주제로 설명 요청
invoke_chain("더블딥")

'더블딥(Double Dip)은 경제학에서 사용되는 용어로, 경기 침체가 두 번에 걸쳐 발생하는 현상을 의미합니다. 일반적으로 경제가 회복세를 보이다가 다시 하락하는 경우를 가리킵니다. 예를 들어, 경제가 일시적으로 회복된 후 다시 불황에 빠지게 되는 상황이 더블딥입니다. 이는 소비자 신뢰 저하, 고용 시장의 둔화, 경제적 불확실성 등 여러 요인으로 인해 발생할 수 있습니다. 더블딥은 정책 입안자들에게 큰 도전 과제가 되며, 경제 회복을 더욱 복잡하게 만듭니다.'

In [11]:
# <랭체인을 사용하여 주제에 대한 간단한 설명 생성 파이프라인 구축>

# 라이브러리 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 미스트랄AI 모델을 사용할 경우 주석 해제
from langchain_mistralai.chat_models import ChatMistralAI

# 주어진 주제에 대해 짧은 설명을 요청하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    "주제 {topic}에 대해 짧은 설명을 해주세요"
)

# 출력 파서를 문자열로 설정
output_parser = StrOutputParser()
# 오픈AI의 gpt-4o 모델을 사용하여 채팅 모델 설정
model = ChatOpenAI(model="gpt-4o")
# 미스트랄AI 모델을 사용할 경우 주석 해제
# model = ChatMistralAI()

# 파이프라인 설정: 주제를 받아 프롬프트를 생성하고, 모델을 통해 응답을 생성한 후 문자열로 파싱
chain = (
    {"topic": RunnablePassthrough()}  # 입력 받은 주제를 그대로 통과시킴
    | prompt                         # 프롬프트 템플릿을 적용
    | model                          # 모델을 사용해 응답 생성
    | output_parser                  # 응답을 문자열로 파싱
)

# "더블딥" 주제로 설명 요청
chain.invoke("위례신사선")

'위례신사선은 대한민국 서울특별시와 경기도 성남시를 연결하는 도시철도 노선입니다. 이 노선은 위례신도시에서 출발하여 서울시내의 여러 지역을 경유한 후, 신사역까지 연결될 계획입니다. 위례신사선은 아직 건설 중이며, 개통되면 위례신도시와 서울 강남 일대 간의 교통 접근성이 크게 개선될 것으로 기대됩니다. 또한, 이 노선은 지역 교통 혼잡을 줄이고 대중교통 이용률을 높이는 데 기여할 것입니다.'